In [4]:
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup


def scrape_tarimorman_news(start_number=153, end_number=7040, output_file="agroforest_ministry_news.xlsx"):
    base_url = "https://www.tarimorman.gov.tr/Haber/{}"

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0.0.0 Safari/537.36"
        )
    }

    session = requests.Session()
    session.headers.update(headers)

    all_rows = []

    for number in range(start_number, end_number + 1):
        url = base_url.format(number)
        print(f"Checking: {url}")

        try:
            response = session.get(url, timeout=15)

            if response.status_code == 200:
                soup = BeautifulSoup(response.text, "html.parser")

                h3_tag = soup.find("h3")
                date_tag = soup.find("span", class_="itemDateCreated")

                title_text = h3_tag.get_text(strip=True) if h3_tag else None
                date_text = date_tag.get_text(strip=True).rstrip("/") if date_tag else None

                all_rows.append({
                    "URL": url,
                    "Date": date_text,
                    "Title": title_text
                    
                })

        except requests.RequestException:
            pass

        time.sleep(0.3)

    df = pd.DataFrame(all_rows)
    df.to_excel(output_file, index=False)

    print(f"\nFinished. Excel saved as: {output_file}")


if __name__ == "__main__":
    scrape_tarimorman_news()

Checking: https://www.tarimorman.gov.tr/Haber/153
Checking: https://www.tarimorman.gov.tr/Haber/154
Checking: https://www.tarimorman.gov.tr/Haber/155
Checking: https://www.tarimorman.gov.tr/Haber/156
Checking: https://www.tarimorman.gov.tr/Haber/157
Checking: https://www.tarimorman.gov.tr/Haber/158
Checking: https://www.tarimorman.gov.tr/Haber/159
Checking: https://www.tarimorman.gov.tr/Haber/160
Checking: https://www.tarimorman.gov.tr/Haber/161
Checking: https://www.tarimorman.gov.tr/Haber/162
Checking: https://www.tarimorman.gov.tr/Haber/163
Checking: https://www.tarimorman.gov.tr/Haber/164
Checking: https://www.tarimorman.gov.tr/Haber/165
Checking: https://www.tarimorman.gov.tr/Haber/166
Checking: https://www.tarimorman.gov.tr/Haber/167
Checking: https://www.tarimorman.gov.tr/Haber/168
Checking: https://www.tarimorman.gov.tr/Haber/169
Checking: https://www.tarimorman.gov.tr/Haber/170
Checking: https://www.tarimorman.gov.tr/Haber/171
Checking: https://www.tarimorman.gov.tr/Haber/172


In [5]:
import pandas as pd


# Read the Excel file
df = pd.read_excel("agroforest_ministry_news.xlsx")

# Check whether "tohum" exists in Title column (case-insensitive)
filtered_df = df[df["Title"].astype(str).str.lower().str.contains("tohum", na=False)].copy()

# Save matching rows to a new Excel file
filtered_df.to_excel("agroforest_ministry_news_seed.xlsx", index=False)

print(filtered_df)
print(f"\nSaved {len(filtered_df)} rows to agroforest_ministry_news_seed.xlsx")

                                           URL         Date  \
83     https://www.tarimorman.gov.tr/Haber/236   9.10.2013    
231    https://www.tarimorman.gov.tr/Haber/384  24.03.2014    
689    https://www.tarimorman.gov.tr/Haber/843  22.01.2016    
705    https://www.tarimorman.gov.tr/Haber/859   4.02.2016    
769    https://www.tarimorman.gov.tr/Haber/923   8.04.2016    
...                                        ...          ...   
6664  https://www.tarimorman.gov.tr/Haber/6822  21.10.2025    
6758  https://www.tarimorman.gov.tr/Haber/6916   4.01.2026    
6799  https://www.tarimorman.gov.tr/Haber/6957   6.02.2026    
6805  https://www.tarimorman.gov.tr/Haber/6963  12.02.2026    
6866  https://www.tarimorman.gov.tr/Haber/7024   3.04.2026    

                                                  Title  
83    Bakan Eker "Dünden Bugüne Sertifikalı Tohum" t...  
231   "Hububat, sertifikalı tohum ve arıcılık destek...  
689   Bakan Çelik, tohumculuk sektörü temsilcileriyl...  
705   Bakan

In [ ]:
import csv
import os
import time
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


FIELDNAMES = ["Number", "URL", "Title", "Date", "Paragraphs"]


def scrape_tarimorman_news_fulltext(start_number=153, end_number=7260, output_file="agroforestministry_news.csv"):
    # NOTE: article body lives in <div class="itemBody">. Searching the whole page
    # for <p> tags (as the original pilot did) also picks up footer contact emails,
    # address/phone, and accessibility-menu boilerplate that appears on every page -
    # confirmed by inspecting Haber/236's raw HTML and the existing 8-row pilot file,
    # where every row has that boilerplate appended to Paragraphs.
    # end_number=7260 confirmed by Orhan (2026-09-11) as the current latest article;
    # the bare /Haber/{number} URL (no slug) resolves fine for it - verified directly.
    base_url = "https://www.tarimorman.gov.tr/Haber/{}"

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0.0.0 Safari/537.36"
        )
    }

    session = requests.Session()
    session.headers.update(headers)
    retries = Retry(total=3, backoff_factor=1, status_forcelist=[500, 502, 503, 504])
    session.mount("https://", HTTPAdapter(max_retries=retries))

    # Resume support: skip Numbers already present in the output file so an
    # interrupted run (crash, network drop) can continue instead of restarting.
    done_numbers = set()
    file_exists = os.path.isfile(output_file)
    if file_exists:
        with open(output_file, "r", encoding="utf-8-sig", newline="") as f:
            for row in csv.DictReader(f):
                done_numbers.add(int(row["Number"]))
        print(f"Resuming: {len(done_numbers)} articles already in {output_file}, skipping those.")

    with open(output_file, "a", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        if not file_exists:
            writer.writeheader()

        for number in range(start_number, end_number + 1):
            if number in done_numbers:
                continue

            url = base_url.format(number)
            print(f"Checking: {url}")

            try:
                response = session.get(url, timeout=15)
            except requests.RequestException as e:
                print(f"  Failed after retries: {e}")
                time.sleep(0.3)
                continue

            if response.status_code == 200:
                soup = BeautifulSoup(response.text, "html.parser")

                content_div = soup.find("div", class_="itemBody")
                h3_tag = soup.find("h3")
                date_tag = soup.find("span", class_="itemDateCreated")

                title_text = h3_tag.get_text(strip=True) if h3_tag else None
                date_text = date_tag.get_text(strip=True).rstrip("/") if date_tag else None

                p_tags = content_div.find_all("p") if content_div else []
                paragraph_texts = [
                    text for p in p_tags
                    if (text := p.get_text(" ", strip=True))
                ]
                concatenated_paragraphs = "\n".join(paragraph_texts) if paragraph_texts else None

                writer.writerow({
                    "Number": number,
                    "URL": url,
                    "Title": title_text,
                    "Date": date_text,
                    "Paragraphs": concatenated_paragraphs,
                })
                f.flush()

            time.sleep(0.3)

    print(f"\nFinished. CSV saved as: {output_file}")


if __name__ == "__main__":
    scrape_tarimorman_news_fulltext()

In [ ]:
import csv
import logging
import re
from collections import Counter

import zeyrek
from nltk.corpus import stopwords

# zeyrek logs each candidate analysis at logging.WARNING level by default ("APPENDING
# RESULT: ..."), which floods stdout during lemmatization - silence it.
logging.getLogger("zeyrek").setLevel(logging.ERROR)

_analyzer = zeyrek.MorphAnalyzer()

TURKISH_STOPWORDS = set(stopwords.words("turkish")) | {
    "dedi", "diye", "konuştu", "belirtti", "söyledi", "ifade", "etti", "yaptı",
    "bir", "olarak", "olduğunu", "üzere", "için", "de", "da", "ile",
}

# Common auxiliary/copula verb lemmas - only become dominant after lemmatization
# merges all their inflected surface forms together (e.g. "olan"/"olarak"/"oldu"/
# "olduğunu" all collapse to "olmak"), so they need filtering at the lemma level
# separately from the surface-form stopword list above.
LEMMA_STOPWORDS = {"olmak", "etmek", "demek", "yapmak", "o"}

_WORD_RE = re.compile(r"[a-zA-ZçÇğĞıİöÖşŞüÜ]+")


def lemmatize_text(text):
    """Lemmatize Turkish text with zeyrek.

    zeyrek returns multiple candidate analyses per surface form when a word is
    morphologically ambiguous, and does NOT order them by likelihood - e.g. for
    "tarım" (agriculture) it returns ['tar', 'tarım'] in that order, so naively
    taking candidates[0] mis-lemmatizes one of this corpus's most important words
    into "tar". Fix: prefer the candidate that exactly equals the raw surface form
    when one exists (usually the correct "bare root, no suffix" reading - confirmed
    against "tarım" and "bin", both mis-lemmatized by candidates[0] alone), falling
    back to candidates[0] otherwise. This does NOT fully solve disambiguation - some
    genuinely ambiguous verb forms (e.g. "ürettikleri" -> "üremek" vs "üretmek")
    still pick an unvalidated first candidate; no context-based disambiguation is
    attempted here.
    """
    if not text:
        return ""
    lemmas = []
    for surface, candidates in _analyzer.lemmatize(text):
        if not candidates:
            lemmas.append(surface.lower())
            continue
        surface_l = surface.lower()
        exact = next((c for c in candidates if c.lower() == surface_l), None)
        lemma = (exact or candidates[0]).lower()
        lemmas.append(lemma)
    return " ".join(lemmas)


def word_frequency_bag(texts, already_lemmatized=False):
    """Raw word-frequency counter ("keyword-bag") over a list of texts, with Turkish
    stopwords and short tokens dropped. Set already_lemmatized=True for text produced
    by lemmatize_text() (whitespace-joined lemmas, also filters LEMMA_STOPWORDS);
    otherwise tokenizes with a simple regex, matching the diagnostic used earlier in
    this strand's exploration.
    """
    counter = Counter()
    for text in texts:
        tokens = (text or "").split() if already_lemmatized else _WORD_RE.findall((text or "").lower())
        skip = TURKISH_STOPWORDS | LEMMA_STOPWORDS if already_lemmatized else TURKISH_STOPWORDS
        for w in tokens:
            if len(w) < 3 or w in skip:
                continue
            counter[w] += 1
    return counter


def lemmatize_csv(src_path, out_path, text_column="Paragraphs", lemma_column="Paragraphs_Lemmatized"):
    """Reads src_path, lemmatizes text_column row by row, writes out_path with an
    added lemma_column. Used on the tohum seed subset (807 rows) as the first test,
    per the agreed subset-testing plan - not run over the full ~7,100-article corpus
    yet.
    """
    with open(src_path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
        fieldnames = list(reader.fieldnames) + [lemma_column]

    for row in rows:
        row[lemma_column] = lemmatize_text(row.get(text_column) or "")

    with open(out_path, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    return rows


if __name__ == "__main__":
    lemmatize_csv(
        "agroforestministry_news_seed.csv",
        "agroforestministry_news_seed_lemmatized.csv",
    )